# 06 — Hyperparameter selection bias under regime shift

**What this notebook is:** the empirical demonstration of the README's Discussion Claim 2 — *Hyperparameter optimization is fragile to regime shift*. Optuna picks different hyperparameters depending on which slice of data it sees during tuning; the resulting models then produce different OOS performance, sometimes worse than just using default hyperparameters.

**Why it matters:** if your CV best HPs aren't actually best in deployment, you have an estimation-error tax on top of an optimization decision — DeMiguel-Garlappi-Uppal (2009) generalized to HP search. The practical implication is: every Optuna-tuned model should be verified against the default-HP baseline on a held-out slice the HPs never saw. In this study, that verification step rejects the 21-day Optuna result entirely.

**Five LightGBM variants on the same v3-curated 14-feature panel:**

| Variant | Target horizon | HP-selection cutoff | Notes |
|---|---|---|---|
| Default HPs (no tuning) | 5-day | — | Hand-picked from a Kaggle hedge-fund notebook |
| HP-leaked Optuna | 5-day | none (all data) | Inflates IC by selection bias |
| Held-out Optuna Split A | 5-day | ≤ 2023-12-31 | First held-out protocol |
| Held-out Optuna Split B | 5-day | ≤ 2024-12-31 | Second held-out protocol, more recent data |
| Default HPs at 21-day | 21-day | — | Apples-to-apples vs 21-day held-out below |
| Held-out Optuna 21-day | 21-day | ≤ 2024-12-31 | Underperforms 21-day default |

Section structure: (1) load chosen HPs from each YAML and put them side-by-side; (2) pull each model's predictions from the store and compute OOS IC on the 2025+ slice; (3) plot OOS IC bar chart showing the held-out-vs-default comparison.


## 0. Setup


In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import yaml

from price_model.data.loaders import load_panel
from price_model.eval.metrics import summarize
from price_model.features.targets import add_forward_excess_return
from price_model.serving.store import PredictionStore

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 5)

# Each sweep maps to a YAML config + a model_id in the store.
SWEEPS = [
    {'label': 'Default HPs (5-day)',         'yaml': 'extended_kaggle_v3_curated.yaml',                       'model_id': 'lightgbm_kaggle_v3_curated',                          'horizon': 5,  'kind': 'default'},
    {'label': 'HP-leaked Optuna (5-day)',    'yaml': 'extended_kaggle_v3_curated_tuned.yaml',                 'model_id': 'lightgbm_kaggle_v3_curated_tuned',                    'horizon': 5,  'kind': 'leaked'},
    {'label': 'Split A (5-day, ≤2023)',      'yaml': 'extended_kaggle_v3_curated_hp_pre20231231.yaml',        'model_id': 'lightgbm_kaggle_v3_curated_hp_pre20231231',           'horizon': 5,  'kind': 'held-out'},
    {'label': 'Split B (5-day, ≤2024)',      'yaml': 'extended_kaggle_v3_curated_hp_pre20241231.yaml',        'model_id': 'lightgbm_kaggle_v3_curated_hp_pre20241231',           'horizon': 5,  'kind': 'held-out'},
    {'label': 'Default HPs (21-day)',        'yaml': 'extended_kaggle_v3_curated_h21.yaml',                   'model_id': 'lightgbm_kaggle_v3_curated_h21',                      'horizon': 21, 'kind': 'default'},
    {'label': '21-day held-out (≤2024)',     'yaml': 'extended_kaggle_v3_curated_h21_hp_pre20241231.yaml',    'model_id': 'lightgbm_kaggle_v3_curated_h21_hp_pre20241231',       'horizon': 21, 'kind': 'held-out'},
]


## 1. Chosen hyperparameters across sweeps

**Why this matters:** if Optuna picked materially different HPs at each cutoff, that's the smoking gun for regime-dependent HP optimization. A stationary problem would have stable HPs across reasonable training-data variations.

The table below loads each YAML config and extracts the LightGBM HP values that Optuna selected (or the hand-picked defaults). Watch `lambda_l1` (sparsity), `lambda_l2` (smoothness), and `learning_rate` — these are the parameters most sensitive to regime.


In [ ]:
CONFIG_DIR = Path('config/experiments')

def load_hps(yaml_path: str) -> dict:
    cfg = yaml.safe_load((CONFIG_DIR / yaml_path).read_text())
    for m in cfg.get('models', []):
        if m.get('class') == 'LightGBMModel':
            return m.get('params', {})
    return {}

HP_KEYS = ['learning_rate', 'num_leaves', 'min_data_in_leaf',
           'lambda_l1', 'lambda_l2', 'n_estimators',
           'feature_fraction', 'bagging_fraction']

rows = []
for s in SWEEPS:
    hps = load_hps(s['yaml'])
    rows.append({'sweep': s['label'], **{k: hps.get(k) for k in HP_KEYS}})
hp_table = pd.DataFrame(rows).set_index('sweep')
with pd.option_context('display.float_format', lambda x: f'{x:.4g}'):
    print(hp_table.T)


**Reading the table:** Optuna's chosen HPs vary by an order of magnitude or more across cutoffs.

- `lambda_l1`: ranges from 0.014 (Split A) to 5.43 (Split B). The same Optuna sweep, with one extra year of training data, picked **390× more L1 regularization**.
- `lambda_l2`: ranges from 0.13 (Split B) to 44.4 (21-day held-out). The 21-day Optuna found that aggressive L2 smoothing was optimal in training — but as we'll see in section 3, that choice did not generalize.
- `learning_rate`: ranges from 0.009 (Split A, very small) to 0.097 (21-day held-out, large). When training-period structure favors slower learning, Optuna picks low; when training-period structure favors faster learning, it picks high. The deployment period is a different regime.

This is the empirical face of regime-dependent HP selection: the optimal HPs are a moving target.


## 2. OOS IC for each sweep on the 2025+ slice

**Why this matters:** the chosen HPs are interesting in themselves, but the actual question is which HPs produced the best OOS performance on the deployment slice. The table below pulls each model's stored predictions, joins to realized 5-day OR 21-day forward returns (matched to the model's target horizon), and reports OOS IC + Sharpe on the 2025+ slice.

**Key comparison:** *Default HPs vs Held-out Optuna at 21-day.* This is the cleanest test — both use the same v3-curated panel, both at 21-day target, but one ran Optuna with a held-out cutoff while the other used hand-picked defaults. If default HPs win, Claim 2 is empirically supported.


In [ ]:
# Load realized targets at both horizons on the PIT-corrected universe.
panel = load_panel(universe='sp500_pit', start='2017-01-01', pit_filter=True)
panel_5 = add_forward_excess_return(panel, horizon_days=5, target_col='y5')
panel_21 = add_forward_excess_return(panel_5, horizon_days=21, target_col='y21')
realized_5 = panel_21.select('date', 'ticker', pl.col('y5').alias('realized'))
realized_21 = panel_21.select('date', 'ticker', pl.col('y21').alias('realized'))

store = PredictionStore(read_only=True)

results = []
for s in SWEEPS:
    preds = store.query(f"""
        SELECT prediction_date AS date, ticker, prediction
        FROM predictions WHERE model_id = '{s['model_id']}'
    """)
    preds = preds.filter(pl.col('date') >= pl.lit('2025-01-01').cast(pl.Date))
    realized = realized_5 if s['horizon'] == 5 else realized_21
    joined = preds.join(realized, on=['date', 'ticker'], how='inner').drop_nulls('realized')
    if joined.height == 0:
        results.append({'sweep': s['label'], 'n_dates': 0, 'ic': float('nan'), 't_stat': float('nan'), 'sharpe': float('nan')})
        continue
    summary = summarize(joined, horizon_days=s['horizon']).as_dict()
    results.append({
        'sweep': s['label'],
        'horizon': s['horizon'],
        'kind': s['kind'],
        'n_dates': summary['n_dates'],
        'ic': summary['information_coefficient'],
        't_stat': summary['ic_t_stat'],
        'sharpe': summary['long_short_sharpe'],
    })

store.close()
ic_table = pd.DataFrame(results).set_index('sweep')
print(ic_table.to_string(float_format=lambda x: f'{x:+.4f}'))


## 3. Visual comparison: held-out Optuna vs default HPs

**Why this matters:** the most important data point in this notebook is whether held-out Optuna *at 21-day horizon* beats default HPs *at 21-day horizon*. The README's Claim 2 says it doesn't — Optuna underperformed defaults by ~0.009 IC. The chart below shows that directly.

**What good optimization looks like:** held-out Optuna ≥ default HPs on the deployment slice.
**What we observe (regime fragility):** held-out Optuna < default HPs at 21-day target.


In [ ]:
# Bar chart: OOS IC by sweep, colored by kind, grouped by horizon.
ic_df = ic_table.reset_index().dropna(subset=['ic'])
color_map = {'default': '#4c72b0', 'leaked': '#ff7f0e', 'held-out': '#2ca02c'}
colors = [color_map[k] for k in ic_df['kind']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(ic_df)), ic_df['ic'], color=colors, edgecolor='black', linewidth=0.5)
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xticks(range(len(ic_df)))
ax.set_xticklabels(ic_df['sweep'], rotation=30, ha='right')
ax.set_ylabel('OOS IC on 2025+ slice')
ax.set_title('LightGBM OOS IC by Optuna sweep — default HPs vs Optuna with various cutoffs')

# Annotate each bar with its IC value
for bar, ic in zip(bars, ic_df['ic']):
    y = ic + 0.0005 if ic >= 0 else ic - 0.0015
    ax.text(bar.get_x() + bar.get_width() / 2, y, f'{ic:+.4f}', ha='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor=color_map['default'], label='Default HPs (hand-picked)'),
    Patch(facecolor=color_map['leaked'], label='Optuna (HP-leaked)'),
    Patch(facecolor=color_map['held-out'], label='Optuna (held-out cutoff)'),
], loc='best')
plt.tight_layout(); plt.show()


## 4. Interpretation

Three takeaways from the chart:

1. **At 21-day target horizon, default HPs (≈ +0.0274) beat held-out Optuna (≈ +0.0187) by ~0.009 IC.** Even with the deployment slice strictly excluded from HP selection, the Optuna sweep picked HPs (high `lambda_l2`, fast learning, fewer trees) optimized for the 2017–2024 training regime that did not generalize.
2. **At 5-day target horizon, the picture is more nuanced.** Split B (≤2024-12-31, more recent training data) scored higher 2025+ OOS than Split A (≤2023-12-31). More recent HPs better calibrated to the regime — but neither beats the HP-leaked baseline, which had access to evaluation-period data during HP selection. The HP-leaked baseline's apparent OOS IC of +0.0046 is itself a regime-overfit artifact.
3. **The DeMiguel-Garlappi-Uppal (2009) analogy holds.** They showed 1/N portfolios beat Markowitz mean-variance optimization out-of-sample because estimation error in optimal weights overwhelms the theoretical benefit of optimization. Here: default HPs beat Optuna-tuned HPs out-of-sample because estimation error in optimal HPs overwhelms the theoretical benefit of HP search.

**Practical recommendation:** verify every Optuna-tuned model against its default-HP baseline on a slice neither saw. If the tuned version doesn't beat default, reject the tuning. In this study, that verification step would reject the 21-day Optuna result entirely.

## Related notebooks / scripts

- The actual sweeps were produced by `scripts/optuna_sweep.py --max-date <date>`. Re-running requires Optuna installed.
- The headline 21-day comparison table (which includes the held-out Optuna LightGBM as one row out of 9) is in the README Executive Summary.
- For the broader stored-prediction diagnostics across the 8 headline models, see `notebooks/01_diagnostics.ipynb`.
